In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FMM (CPU) sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, int src)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    heap.push(Entry(u[src], src));                   // la sorgente e' il primo Trial
    state[src] = ATTIVO;

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}
// Errore relativo ||u - U|| / ||U|| rispetto alla soluzione esatta U = distanza dal centro
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h, z = -1.0 + k * h;   // coordinate del nodo (k, i, j)
            double U = std::sqrt(x * x + y * y + z * z);        // soluzione esatta
            double d = u[(k * N + i) * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
            }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}

In [ ]:
%%writefile node_fim.cuh
#pragma once           // il file viene incluso una sola volta
// FIM a nodi (Alg. 6 della tesi) sulla GPU: lista attiva di nodi, un thread per nodo, niente shared memory
// Versione con maschera (ostacoli fermi a INF) e piu' sorgenti (da BlockFIM_vs_FIM_FM2_3D.ipynb)
// Si include in fim3d.cu dopo la definizione di CUDA_CHECK; INF e solve_quadratic3 vengono da fmm3d.h
#include <vector>
#include <utility>         // std::swap
#include <cuda_runtime.h>
#include "fmm3d.h"         // INF e solve_quadratic3

// Nuovo u del nodo x, leggendo i 6 vicini dalla memoria globale (INF fuori dalla griglia)
__device__ float solve_global(const float* u, const float* F, int x, int N, float h)
{
    int k = x / (N * N), i = (x / N) % N, j = x % N;             // strato, riga e colonna del nodo
    float a = fminf(j > 0 ? u[x - 1] : INF, j < N - 1 ? u[x + 1] : INF);              // vicini lungo x
    float b = fminf(i > 0 ? u[x - N] : INF, i < N - 1 ? u[x + N] : INF);              // vicini lungo y
    float c = fminf(k > 0 ? u[x - N * N] : INF, k < N - 1 ? u[x + N * N] : INF);      // vicini lungo z
    return fminf(u[x], solve_quadratic3(a, b, c, F[x], h));      // return min(u_x, u)
}

// Alg. 6, prima parte: ogni nodo x della lista A si aggiorna (p <- u(x), q <- g(x), u(x) <- q)
__global__ void node_update(const int* list, int count, float* u, const float* F,
                            int N, float h, float eps, unsigned char* conv)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;               // un thread per nodo della lista
    if (t >= count) return;
    int x = list[t];
    float p = u[x];
    float q = solve_global(u, F, x, N, h);
    u[x] = q;
    conv[t] = fabsf(p - q) < eps;                                // |p - q| < eps: nodo convergente
}

// Alg. 6, seconda parte: un nodo non convergente resta in A; uno convergente esce
// e prova i vicini fuori da A: se un vicino migliora, entra nella nuova lista
__global__ void node_neighbors(const int* list, int count, const unsigned char* conv,
                               const unsigned char* in_list, const unsigned char* mask, float* u, const float* F, int N, float h,
                               int* next_flag, int* next_list, int* next_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t >= count) return;
    int x = list[t];
    if (!conv[t]) {                                              // non convergente: resta attivo
        if (atomicExch(&next_flag[x], 1) == 0) next_list[atomicAdd(next_count, 1)] = x;
        return;
    }
    int k = x / (N * N), i = (x / N) % N, j = x % N;
    int  nb[6] = { x - 1,  x + 1,     x - N, x + N,     x - N * N, x + N * N };   // sinistra, destra, sotto, sopra, davanti, dietro
    bool ok[6] = { j > 0, j < N - 1, i > 0, i < N - 1, k > 0,     k < N - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d] || in_list[nb[d]] || !mask[nb[d]]) continue;  // fuori griglia, gia' in A, oppure ostacolo
        int y = nb[d];
        float p = u[y];
        float q = solve_global(u, F, y, N, h);
        if (q < p) {                                             // p > q: il vicino migliora
            u[y] = q;
            if (atomicExch(&next_flag[y], 1) == 0)               // aggiungi a A (una volta sola)
                next_list[atomicAdd(next_count, 1)] = y;
        }
    }
}

// Aggiornamento dei flag di appartenenza: la lista vecchia esce
__global__ void clear_in_list(const int* list, int count, unsigned char* in_list)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) in_list[list[t]] = 0;
}

// e la nuova entra (next_flag torna a 0 per l'iterazione successiva)
__global__ void set_in_list(const int* list, int count, unsigned char* in_list, int* next_flag)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) { in_list[list[t]] = 1; next_flag[list[t]] = 0; }
}

// FIM a nodi (Alg. 6) sulla GPU: restituisce u, il tempo del ciclo (ms) e il numero di iterazioni
int node_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
             const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, float* ms)
{
    const int n = N * N * N;                                     // numero di nodi
    const size_t bytes = (size_t)n * sizeof(float);
    const int threads = 256;

    // Memoria sulla GPU
    float *d_u, *d_F; int *d_list, *d_next, *d_count, *d_flag; unsigned char *d_conv, *d_in, *d_mask;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_list, n * sizeof(int)));            // lista A attuale
    CUDA_CHECK(cudaMalloc(&d_next, n * sizeof(int)));            // lista A del passo successivo
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_flag, n * sizeof(int)));            // 1 = gia' messo nella nuova lista
    CUDA_CHECK(cudaMalloc(&d_conv, n));                          // convergenza dei nodi della lista
    CUDA_CHECK(cudaMalloc(&d_in, n));                            // 1 = nodo in A
    CUDA_CHECK(cudaMalloc(&d_mask, n));                          // 1 = libero, 0 = ostacolo
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), n, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_flag, 0, n * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_in, 0, n));

    // All'inizio A contiene le sorgenti
    int count = (int)srcs.size();
    CUDA_CHECK(cudaMemcpy(d_list, srcs.data(), count * sizeof(int), cudaMemcpyHostToDevice));
    set_in_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, d_in, d_flag);
    CUDA_CHECK(cudaGetLastError());

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                          // si misura solo il ciclo
    int iter = 0;
    while (count > 0) {                                          // while A != vuoto
        ++iter;
        const int blocks = (count + threads - 1) / threads;
        node_update<<<blocks, threads>>>(d_list, count, d_u, d_F, N, h, eps, d_conv);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        node_neighbors<<<blocks, threads>>>(d_list, count, d_conv, d_in, d_mask, d_u, d_F, N, h,
                                            d_flag, d_next, d_count);
        CUDA_CHECK(cudaGetLastError());
        clear_in_list<<<blocks, threads>>>(d_list, count, d_in);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));   // dimensione della nuova A
        if (count > 0) {
            set_in_list<<<(count + threads - 1) / threads, threads>>>(d_next, count, d_in, d_flag);
            CUDA_CHECK(cudaGetLastError());
        }
        std::swap(d_list, d_next);                               // la nuova lista diventa quella attuale
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sulla CPU e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));
    CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_next)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_flag)); CUDA_CHECK(cudaFree(d_conv)); CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_mask));
    return iter;
}

In [ ]:
%%writefile block_fim3d.cuh
// Block FIM (Alg. 7) e FIM a nodi (Alg. 6) della sfera, copiati da 1_sfera/BFIM_vs_FMM_vs_FIM.ipynb (stessi parametri)
#pragma once
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include "fmm3d.h"           // Fast Marching

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)
#include "node_fim.cuh"     // FIM a nodi (GPU, Alg. 6): usa CUDA_CHECK, quindi va incluso dopo la sua definizione

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo

// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, int src, float eps, unsigned char* Cb_out)
{
    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || (idx == src);             // nodo da non aggiornare: fuori dalla griglia o sorgente (u = 0)
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q;
        if (fixed) {q = p;}
        else{
        // tx, ty, tz: 0,...,7; +1: nella shared memory c'è l'halo
        q = solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        }    
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break; // se nessun nodo è cambiato esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
}
// Step1: ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* active_list, float* u, const float* F,
                                    int N, int nbx, float h, int src, int niter,
                                    float eps, unsigned char* Cb_step1)
{
    // blockIdx è il numero del blocco CUDA nel lancio in corso
    process_block(active_list[blockIdx.x], niter, u, F, N, nbx, h, src, eps, Cb_step1);
    // (N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Step 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* is_candidate,
                                        float* u, const float* F, int N, int nbx, float h,
                                        int src, float eps, unsigned char* Cb)
{
    // scarta blocchi CUDA in eccesso
    if (blockIdx.x >= *cand_count) return;
    int b = cand_list[blockIdx.x];
    // blocchi candidati: niter = 1 -> con la nuova informazione dal blocco convergente, questo vicino cambia?
    process_block(b, 1, u, F, N, nbx, h, src, eps, Cb);
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) is_candidate[b] = 0;   // pronto per il giro successivo
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              int src, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    const int sk = src / (N * N), si = (src / N) % N, sj = src % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;

    // Allocazione memoria sulla GPU
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb_step1, *d_in_list; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes)); CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks)); CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_active_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, d_u, d_F, N, nbx, h, src, niter, eps, d_Cb_step1);    // si lanciano n_active blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_candidate_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_is_candidate,
                                                                                    d_u, d_F, N, nbx, h, src, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

In [ ]:
%%writefile jacobi_fim.cuh
// FIM-Jacobi senza lista di G. Cirillo (Tesi-Magistrale, GoldStandard.ipynb: FIM_senzaLista_raw), riscritto in CUDA 3D.
// Stessa logica del suo codice CuPy: a ogni iterazione TUTTI i nodi si aggiornano con Godunov usando i valori
// dell'iterazione prima (Jacobi), nuovo = min(vecchio, Godunov) come il suo cp.minimum, due buffer che si alternano,
// convergenza controllata solo ogni CONTROLLO iterazioni (lui: check_every = 25) con max|vecchio - nuovo| < eps.
// Il risolutore locale e' il nostro solve_quadratic3 (lo schema di Godunov del primo ordine esteso al 3D).
#pragma once
#include <vector>
#include <utility>          // std::swap
#include <cuda_runtime.h>
#include "fmm3d.h"          // INF e solve_quadratic3

const int CONTROLLO = 25;   // ogni quante iterazioni si controlla la convergenza (come check_every del suo codice)

// Un thread per nodo: nuovo valore del nodo x dai 6 vicini dell'iterazione precedente
__global__ void jacobi_update(const float* cur, const float* F, float* nxt, int N, float h, float eps, int* cambiato)
{
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    if (x >= N * N * N) return;                                   // thread in eccesso
    const float p = cur[x];
    if (p == 0.0f) { nxt[x] = p; return; }                        // sorgente: non si tocca mai (come nel suo kernel)
    const int j = x % N, i = (x / N) % N, k = x / (N * N);
    const float a = fminf(j > 0 ? cur[x - 1] : INF, j < N - 1 ? cur[x + 1] : INF);              // vicini lungo x
    const float b = fminf(i > 0 ? cur[x - N] : INF, i < N - 1 ? cur[x + N] : INF);              // vicini lungo y
    const float c = fminf(k > 0 ? cur[x - N * N] : INF, k < N - 1 ? cur[x + N * N] : INF);      // vicini lungo z
    const float q = fminf(p, solve_quadratic3(a, b, c, F[x], h));                               // min(vecchio, Godunov)
    nxt[x] = q;
    if (cambiato && p - q >= eps) *cambiato = 1;                  // solo nelle iterazioni di controllo: qualcuno cambia ancora
}

// Restituisce il numero di iterazioni; in ms il tempo del solo ciclo (eventi CUDA), come per gli altri due metodi
int jacobi_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h, float eps, float* ms)
{
    const int n = N * N * N;
    const size_t bytes = (size_t)n * sizeof(float);
    float *d_cur, *d_nxt, *d_F; int* d_cambiato;
    CUDA_CHECK(cudaMalloc(&d_cur, bytes)); CUDA_CHECK(cudaMalloc(&d_nxt, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_cambiato, sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d_cur, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    const int threads = 256, blocchi = (n + threads - 1) / threads;   // come il suo threads_per_block = 256
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));
    int it = 0;
    while (true) {
        const bool controlla = (it % CONTROLLO == 0);
        if (controlla) CUDA_CHECK(cudaMemset(d_cambiato, 0, sizeof(int)));
        jacobi_update<<<blocchi, threads>>>(d_cur, d_F, d_nxt, N, h, eps, controlla ? d_cambiato : nullptr);
        CUDA_CHECK(cudaGetLastError());
        std::swap(d_cur, d_nxt);                                  // il nuovo diventa il vecchio
        ++it;
        if (controlla) {                                          // sincronizzazione GPU -> CPU solo ogni CONTROLLO iterazioni
            int cambiato = 0;
            CUDA_CHECK(cudaMemcpy(&cambiato, d_cambiato, sizeof(int), cudaMemcpyDeviceToHost));
            if (!cambiato) break;
        }
    }
    CUDA_CHECK(cudaEventRecord(stop)); CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaMemcpy(u.data(), d_cur, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_cur)); CUDA_CHECK(cudaFree(d_nxt)); CUDA_CHECK(cudaFree(d_F)); CUDA_CHECK(cudaFree(d_cambiato));
    return it;
}

In [ ]:
%%writefile confronto.cu
// Tempi sulla sfera (F = 1, sorgente al centro): FIM-Jacobi di Cirillo, FIM a nodi e Block FIM, tutti sulla GPU.
// Ogni tempo = media di 3 misure del solo ciclo di calcolo (eventi CUDA). Si stampano solo i tempi; la soluzione dei tre
// metodi viene comunque confrontata, e se differisce si stampa un avviso (un tempo veloce ma sbagliato non vale).
#include <cstdio>
#include <vector>
#include <cmath>
#include "block_fim3d.cuh"   // CUDA_CHECK, fmm3d.h, FIM a nodi e Block FIM
#include "jacobi_fim.cuh"    // FIM-Jacobi

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s\n", prop.name);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza (uguale per i tre metodi)
    const int niter = 24;                                    // iterazioni locali del Block FIM (come nel notebook della sfera)
    const int ripetizioni = 3;                               // ogni tempo = la media di 3 misure
    printf("Sfera, tempi [ms] = media di %d misure del solo ciclo di calcolo\n", ripetizioni);
    printf("    N   t FIM-Jacobi   t FIM a nodi   t Block FIM   Jacobi / FIM   Jacobi / BFIM\n");
    for (int Nk : { 33, 65, 129, 193, 257, 385 }) {
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo [-1, 1]^3
        const int n = Nk * Nk * Nk;
        const int src = ((Nk / 2) * Nk + Nk / 2) * Nk + Nk / 2;   // sorgente al centro
        const std::vector<float> Fk(n, 1.0f);                // mezzo omogeneo
        const std::vector<unsigned char> mask(n, 1);         // nessun ostacolo (serve al FIM a nodi)
        std::vector<float> u0(n, INF);
        u0[src] = 0.0f;
        std::vector<float> uj, un, ub;
        double tj = 0.0, tn = 0.0, tb = 0.0;
        for (int r = 0; r < ripetizioni; ++r) {
            float ms = 0.0f;
            uj = u0; jacobi_fim(uj, Fk, Nk, hk, eps, &ms); tj += ms;                    // FIM-Jacobi (Cirillo)
            un = u0; node_fim(un, Fk, Nk, hk, { src }, mask, eps, &ms); tn += ms;        // FIM a nodi (Alg. 6)
            ub = u0; block_fim(ub, Fk, Nk, hk, src, eps, niter, &ms); tb += ms;          // Block FIM (Alg. 7)
        }
        tj /= ripetizioni; tn /= ripetizioni; tb /= ripetizioni;
        float diff = 0.0f;                                   // i tre metodi devono dare la stessa u
        for (int q = 0; q < n; ++q) diff = fmaxf(diff, fmaxf(fabsf(uj[q] - ub[q]), fabsf(un[q] - ub[q])));
        if (diff > 1e-4f) printf("ATTENZIONE (N = %d): i metodi differiscono fino a %.2e\n", Nk, diff);
        printf("%5d   %12.2f   %12.2f   %11.2f   %12.1f   %13.1f\n", Nk, tj, tn, tb, tj / tn, tj / tb);
    }
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 confronto.cu -o confronto && ./confronto